# Sinus_CFD — nnU-Net training on NasalSeg (Google Colab)

Trains a 3D nnU-Net v2 segmentation model on the NasalSeg dataset (130 head CTs,
labels: background, left/right nasal cavity, nasopharynx, left/right maxillary
sinus). Goal: give the classical HU-threshold pipeline in `sinus_cfd.pipeline`
a learned alternative that can tell nasal-cavity air apart from sinus air
(see `docs/stage1_segmentation_baseline.md` for why that distinction is the
current bottleneck).

**Before running this notebook:**
1. On your machine: `py -3.12 scripts/download_nasalseg.py` then
   `py -3.12 scripts/prepare_nnunet_nasalseg.py --nasalseg-root data`
2. Zip `data/nnUNet_raw/Dataset501_NasalSeg/` and upload the zip to your
   Google Drive (e.g. `MyDrive/sinus_cfd/NasalSeg_nnUNet_raw.zip`).
3. Runtime → Change runtime type → GPU (T4 is fine to start; A100 if you have
   Colab Pro/Pro+ and want it faster).

Colab sessions are ephemeral and can disconnect after ~12h (free tier: often
less, and idles out after ~90 min of inactivity). This notebook checkpoints
training results back to Drive after each run so a disconnect doesn't lose
progress — rerun the training cell with `--c` (continue) if it stops early.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install nnU-Net v2

In [ ]:
!pip install -q nnunetv2>=2.5.0

## 3. Mount Drive and stage the dataset locally

nnU-Net does heavy small-file I/O during preprocessing/training; Drive-mounted
storage is much slower for that than Colab's local disk, so copy the zip in
and extract it to `/content` rather than pointing `nnUNet_raw` at Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Edit this path if you uploaded the zip somewhere else in Drive
DRIVE_ZIP = '/content/drive/MyDrive/sinus_cfd/NasalSeg_nnUNet_raw.zip'
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/sinus_cfd/nnUNet_results'

In [ ]:
import os
os.makedirs('/content/nnUNet_data/nnUNet_raw', exist_ok=True)
os.makedirs('/content/nnUNet_data/nnUNet_preprocessed', exist_ok=True)
os.makedirs('/content/nnUNet_data/nnUNet_results', exist_ok=True)
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

!unzip -q -o "$DRIVE_ZIP" -d /content/nnUNet_data/nnUNet_raw
!ls /content/nnUNet_data/nnUNet_raw

## 4. Set nnU-Net environment variables

In [ ]:
import os
os.environ['nnUNet_raw'] = '/content/nnUNet_data/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_data/nnUNet_preprocessed'
os.environ['nnUNet_results'] = '/content/nnUNet_data/nnUNet_results'
print(os.environ['nnUNet_raw'])
print(os.environ['nnUNet_preprocessed'])
print(os.environ['nnUNet_results'])

## 5. Plan and preprocess (dataset id 501 = NasalSeg)

This resamples all 130 cases to nnU-Net's chosen spacing and computes the
network configuration. Takes a few minutes for a dataset this size.

In [ ]:
!nnUNetv2_plan_and_preprocess -d 501 --verify_dataset_integrity

## 6. Train fold 0 of 3d_fullres

Default is 1000 epochs. If the runtime disconnects partway through, rerun
this same cell with `--c` appended to resume from the last checkpoint
instead of restarting.

For an MVP, training **fold 0 only** (not the full 5-fold CV) is enough to
get a usable model and a held-out validation Dice; add folds 1-4 later if
you want a proper cross-validated estimate.

In [ ]:
!nnUNetv2_train 501 3d_fullres 0

## 7. Copy trained weights back to Drive

Colab's local disk does not persist across sessions — do this before the
runtime is recycled.

In [ ]:
!rsync -av /content/nnUNet_data/nnUNet_results/ "$DRIVE_RESULTS_DIR/"
print('Synced to', DRIVE_RESULTS_DIR)

## 8. (Optional) Validation Dice summary

nnU-Net writes per-case validation metrics under
`nnUNet_results/Dataset501_NasalSeg/.../fold_0/validation/summary.json`.

In [ ]:
import glob, json
summaries = glob.glob('/content/nnUNet_data/nnUNet_results/Dataset501_NasalSeg/**/summary.json', recursive=True)
print(summaries)
if summaries:
    s = json.load(open(summaries[0]))
    print(json.dumps(s.get('foreground_mean', s), indent=2))

## Next step back in the repo

Download `nnUNet_results/` from Drive to `data/nnUNet_results/` locally (or
just the best checkpoint), then run inference with:

```
nnUNetv2_predict -i INPUT_FOLDER -o OUTPUT_FOLDER -d 501 -c 3d_fullres -f 0
```

and compare its Dice against the same NasalSeg ground truth using
`scripts/evaluate_nasalseg_dice.py`'s methodology (per-case Dice vs labels
1-3), to see whether the learned model actually separates nasal-cavity air
from sinus air where the classical threshold pipeline couldn't.